In [ ]:
# Resolve defaults from this project's root, independently of the working directory.
from pathlib import Path as _KSPath
import sys as _ks_sys
from functools import partial as _ks_partial
_ks_start = _KSPath.cwd()
_ks_root = next((p for p in (_ks_start, *_ks_start.parents) if (p / 'kinspec_paths.py').is_file()), None)
if _ks_root is None:
    raise RuntimeError('Cannot locate the KinSPEC project root containing kinspec_paths.py. Start the notebook kernel from the project directory or a subdirectory.')
_ks_sys.path.insert(0, str(_ks_root))
from kinspec_paths import ROOT as KINSPEC_ROOT, project_path as _ks_project_path
_kp = _ks_partial(_ks_project_path, script_dir='model_interpretability')
import os as _ks_os
_ks_os.chdir(KINSPEC_ROOT / 'model_interpretability')



In [3]:
import os
import math
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# 1. Input / output setting
# =========================
input_csv = _kp('feature_embedding/Kinase_test_MCA/human_482_typical_kinase_1955bit.csv')
output_dir = _kp('model_interpretability/entropy_analysis_output')
os.makedirs(output_dir, exist_ok=True)

# =========================
# 2. Property names (fixed order)
# =========================
property_names = [
    # side chain type
    "Aliphatic", "Aromatic","Sulfur", "Imino", "Amide", "Hydroxyl", "Acidic", "Basic",
    # Tertiary structure preference
    "Exposed", "Buried", "Intermediate",
    # Functional property
    "Func_Acidic", "Func_Basic", "Hydrophobic", "Polar",
    # Secondary Structure preference
    "Alpha_helix", "Beta_strand", "Coil",
    # Size
    "Tiny", "Small", "Medium", "Large", "Very_Large"
]
assert len(property_names) == 23, "Property name count must be 23."

# =========================
# 3. KLIFS functional region mapping
# =========================
region_ranges = [
    ("beta-strand-I",   1, 3),
    ("glycine-rich loop", 4, 9),
    ("beta-strand-II",  10, 13),
    ("beta-strand-III", 14, 19),
    ("alpha-C helix",   20, 30),
    ("back loop",       31, 37),
    ("beta-strand-IV",  38, 41),
    ("beta-strand-V",   42, 44),
    ("gatekeeper",      45, 45),
    ("hinge region",    46, 48),
    ("linker",          49, 52),
    ("alpha-D helix",   53, 59),
    ("alpha-E helix",   60, 64),
    ("beta-strand-VI",  65, 67),
    ("catalytic loop",  68, 75),
    ("beta-strand-VII", 76, 78),
    ("beta-strand-VIII",79, 79),
    ("xDFG",            80, 83),
    ("activation loop", 84, 85),
]

def get_region_name(position):
    for region_name, start, end in region_ranges:
        if start <= position <= end:
            return region_name
    return "Unknown"

# =========================
# 4. Helper functions
# =========================
def shannon_entropy_from_counts(counts):
    counts = np.array(counts, dtype=float)
    total = counts.sum()
    if total == 0:
        return 0.0
    probs = counts[counts > 0] / total
    return -np.sum(probs * np.log(probs))

def binary_entropy(p):
    if p <= 0.0 or p >= 1.0:
        return 0.0
    return -(p * math.log(p) + (1 - p) * math.log(1 - p))

def clean_klifs_sequence(seq):
    if pd.isna(seq):
        return None
    return str(seq).strip().replace(" ", "")

def parse_binary_string(binary_value):
    if pd.isna(binary_value):
        return None

    s = str(binary_value).strip()

    if set(s).issubset({"0", "1"}) and len(s) == 1955:
        return [int(ch) for ch in s]

    if s.startswith("[") and s.endswith("]"):
        try:
            vals = ast.literal_eval(s)
            return [int(x) for x in vals]
        except Exception:
            pass

    if "," in s:
        parts = [x.strip() for x in s.split(",") if x.strip() != ""]
        try:
            return [int(x) for x in parts]
        except Exception:
            pass

    parts = s.split()
    if len(parts) > 1:
        try:
            return [int(x) for x in parts]
        except Exception:
            pass

    raise ValueError(f"Unrecognized Binary_string format: {s[:100]} ...")

def add_region_background(ax, region_ranges, alpha=0.10):
    """
    Add alternating shaded regions to the plot background.
    """
    for i, (region_name, start, end) in enumerate(region_ranges):
        if i % 2 == 0:
            ax.axvspan(start - 0.5, end + 0.5, alpha=alpha)

    # draw region boundaries
    for region_name, start, end in region_ranges:
        ax.axvline(start - 0.5, linestyle="--", linewidth=0.5, alpha=0.4)

def add_region_labels(ax, region_ranges, y_pos=1.02, rotation=90, fontsize=8):
    """
    Add region names at top of plot.
    """
    ylim = ax.get_ylim()
    y = ylim[1] * y_pos
    for region_name, start, end in region_ranges:
        x = (start + end) / 2
        ax.text(x, y, region_name, rotation=rotation, ha="center", va="bottom", fontsize=fontsize)

# =========================
# 5. Load data
# =========================
df = pd.read_csv(input_csv)
print(f"Loaded rows: {len(df)}")
print(f"Columns: {list(df.columns)}")

required_cols = ["KLIFS", "Binary_string"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

# =========================
# 6. Parse KLIFS sequences
# =========================
df["KLIFS_clean"] = df["KLIFS"].apply(clean_klifs_sequence)

valid_klifs_mask = df["KLIFS_clean"].notna() & (df["KLIFS_clean"].str.len() == 85)
invalid_klifs_count = (~valid_klifs_mask).sum()
if invalid_klifs_count > 0:
    print(f"Warning: {invalid_klifs_count} rows have invalid KLIFS length and will be excluded.")

df_klifs = df.loc[valid_klifs_mask].copy()
print(f"Rows retained for KLIFS analysis: {len(df_klifs)}")

residue_matrix = np.array([list(seq) for seq in df_klifs["KLIFS_clean"].tolist()])

# =========================
# 7. Residue-level entropy
# =========================
residue_entropy_records = []

for pos in range(85):
    pos1 = pos + 1
    residues = residue_matrix[:, pos]
    counts = pd.Series(residues).value_counts().sort_index()
    entropy_val = shannon_entropy_from_counts(counts.values)

    row = {
        "KLIFS_position": pos1,
        "Functional_region": get_region_name(pos1),
        "Residue_entropy": entropy_val,
        "Unique_residue_count": int(counts.shape[0]),
        "Most_common_residue": counts.idxmax(),
        "Most_common_residue_freq": counts.max() / counts.sum()
    }

    for aa, cnt in counts.items():
        row[f"AA_{aa}_count"] = cnt
        row[f"AA_{aa}_freq"] = cnt / counts.sum()

    residue_entropy_records.append(row)

residue_entropy_df = pd.DataFrame(residue_entropy_records)
residue_entropy_csv = os.path.join(output_dir, "residue_entropy_by_KLIFS_position.csv")
residue_entropy_df.to_csv(residue_entropy_csv, index=False)
print(f"Saved residue entropy table: {residue_entropy_csv}")

# =========================
# 8. Parse property binary strings
# =========================
parsed_binary_list = []
valid_binary_indices = []

for idx, val in df_klifs["Binary_string"].items():
    try:
        bits = parse_binary_string(val)
        if len(bits) != 1955:
            print(f"Warning: row index {idx} has binary length {len(bits)} != 1955, excluded.")
            continue
        parsed_binary_list.append(bits)
        valid_binary_indices.append(idx)
    except Exception as e:
        print(f"Warning: row index {idx} failed to parse Binary_string, excluded. Error: {e}")

df_prop = df_klifs.loc[valid_binary_indices].copy()
print(f"Rows retained for property analysis: {len(df_prop)}")

property_array = np.array(parsed_binary_list, dtype=int)
property_tensor = property_array.reshape(len(df_prop), 85, 23)

# =========================
# 9. Property-level entropy
# =========================
property_entropy_position_records = []
property_entropy_detailed_records = []

for pos in range(85):
    pos1 = pos + 1
    bit_entropies = []
    bit_activation_freqs = []

    detailed_row = {
        "KLIFS_position": pos1,
        "Functional_region": get_region_name(pos1)
    }

    for bit_idx in range(23):
        bit_values = property_tensor[:, pos, bit_idx]
        p1 = bit_values.mean()
        ent = binary_entropy(p1)

        bit_entropies.append(ent)
        bit_activation_freqs.append(p1)

        pname = property_names[bit_idx]
        detailed_row[f"{pname}_entropy"] = ent
        detailed_row[f"{pname}_p1"] = p1

    property_entropy_position_records.append({
        "KLIFS_position": pos1,
        "Functional_region": get_region_name(pos1),
        "Property_entropy_mean": float(np.mean(bit_entropies)),
        "Property_entropy_sum": float(np.sum(bit_entropies)),
        "Property_entropy_std": float(np.std(bit_entropies)),
        "Property_activation_mean": float(np.mean(bit_activation_freqs))
    })

    property_entropy_detailed_records.append(detailed_row)

property_entropy_position_df = pd.DataFrame(property_entropy_position_records)
property_entropy_detailed_df = pd.DataFrame(property_entropy_detailed_records)

property_entropy_position_csv = os.path.join(output_dir, "property_entropy_summary_by_KLIFS_position.csv")
property_entropy_detailed_csv = os.path.join(output_dir, "property_entropy_detailed_by_KLIFS_position.csv")

property_entropy_position_df.to_csv(property_entropy_position_csv, index=False)
property_entropy_detailed_df.to_csv(property_entropy_detailed_csv, index=False)

print(f"Saved property entropy summary: {property_entropy_position_csv}")
print(f"Saved property entropy detail: {property_entropy_detailed_csv}")

# =========================
# 10. Merge residue + property summary
# =========================
merged_df = residue_entropy_df.merge(
    property_entropy_position_df,
    on=["KLIFS_position", "Functional_region"],
    how="inner"
)

merged_csv = os.path.join(output_dir, "residue_property_entropy_merged.csv")
merged_df.to_csv(merged_csv, index=False)
print(f"Saved merged summary: {merged_csv}")

# =========================
# 11. Region-level summary
# =========================
region_summary_df = merged_df.groupby("Functional_region", as_index=False).agg(
    Position_count=("KLIFS_position", "count"),
    Position_start=("KLIFS_position", "min"),
    Position_end=("KLIFS_position", "max"),
    Residue_entropy_mean=("Residue_entropy", "mean"),
    Residue_entropy_std=("Residue_entropy", "std"),
    Residue_entropy_sum=("Residue_entropy", "sum"),
    Property_entropy_mean=("Property_entropy_mean", "mean"),
    Property_entropy_sum=("Property_entropy_sum", "sum")
)

# keep original region order
region_order_df = pd.DataFrame({"Functional_region": [x[0] for x in region_ranges]})
region_summary_df = region_order_df.merge(region_summary_df, on="Functional_region", how="left")

region_summary_csv = os.path.join(output_dir, "functional_region_entropy_summary.csv")
region_summary_df.to_csv(region_summary_csv, index=False)
print(f"Saved functional region summary: {region_summary_csv}")

# =========================
# 12. Plot 1: residue entropy across positions with region background
# =========================
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(residue_entropy_df["KLIFS_position"], residue_entropy_df["Residue_entropy"], marker="o", linewidth=1)
ax.set_xlabel("KLIFS position")
ax.set_ylabel("Residue Shannon entropy")
ax.set_title("Residue-level entropy across KLIFS positions")
ax.grid(alpha=0.3)

add_region_background(ax, region_ranges, alpha=0.08)
plt.tight_layout()

plot1_path = os.path.join(output_dir, "residue_entropy_plot_with_regions.png")
plt.savefig(plot1_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved plot: {plot1_path}")

# plot 1-2 : residue entropy sorted by entropy (descending)
sorted_df = residue_entropy_df.sort_values("Residue_entropy", ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(16, 6))
bars = ax.bar(
    range(len(sorted_df)),
    sorted_df["Residue_entropy"],
    color="steelblue",
    edgecolor="none"
)

# X-axis labels: position number and functional region
x_labels = [
    # f"{int(row.KLIFS_position)}\n({row.Functional_region})"
    f"{int(row.KLIFS_position)}"
    for _, row in sorted_df.iterrows()
]
ax.set_xticks(range(len(sorted_df)))
ax.set_xticklabels(x_labels, rotation=90, fontsize=6)

ax.set_xlabel("KLIFS position (sorted by entropy)")
ax.set_ylabel("Residue Shannon entropy")
ax.set_title("Residue-level entropy ranked by variability (high → low)")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()

plot1b_path = os.path.join(output_dir, "residue_entropy_sorted_descending.png")
plt.savefig(plot1b_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved plot: {plot1b_path}")


# =========================
# 13. Plot 2: property entropy across positions with region background
# =========================
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(property_entropy_position_df["KLIFS_position"],
        property_entropy_position_df["Property_entropy_mean"],
        marker="o", linewidth=1)
ax.set_xlabel("KLIFS position")
ax.set_ylabel("Mean property Shannon entropy")
ax.set_title("Property-level mean entropy across KLIFS positions")
ax.grid(alpha=0.3)

add_region_background(ax, region_ranges, alpha=0.08)
plt.tight_layout()

plot2_path = os.path.join(output_dir, "property_mean_entropy_plot_with_regions.png")
plt.savefig(plot2_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved plot: {plot2_path}")

# =========================
# 14. Plot 3: residue vs property entropy comparison
# =========================
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(merged_df["KLIFS_position"], merged_df["Residue_entropy"], label="Residue entropy", linewidth=1.5)
ax.plot(merged_df["KLIFS_position"], merged_df["Property_entropy_mean"], label="Mean property entropy", linewidth=1.5)
ax.set_xlabel("KLIFS position")
ax.set_ylabel("Entropy")
ax.set_title("Residue entropy vs property entropy across KLIFS positions")
ax.legend()
ax.grid(alpha=0.3)

add_region_background(ax, region_ranges, alpha=0.08)
plt.tight_layout()

plot3_path = os.path.join(output_dir, "residue_vs_property_entropy_plot_with_regions.png")
plt.savefig(plot3_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved plot: {plot3_path}")

# =========================
# 15. Plot 4: property entropy heatmap
# =========================
heatmap_df = property_entropy_detailed_df.set_index("KLIFS_position")
heatmap_matrix = heatmap_df[[f"{p}_entropy" for p in property_names]].values

plt.figure(figsize=(16, 10))
im = plt.imshow(heatmap_matrix.T, aspect="auto", interpolation="nearest")
plt.colorbar(im, label="Entropy")
plt.yticks(range(len(property_names)), property_names)
plt.xticks(
    ticks=np.arange(0, 85, 5),
    labels=np.arange(1, 86, 5)
)
plt.xlabel("KLIFS position")
plt.ylabel("Physicochemical property")
plt.title("Property-level entropy heatmap")
plt.tight_layout()

plot4_path = os.path.join(output_dir, "property_entropy_heatmap.png")
plt.savefig(plot4_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved plot: {plot4_path}")

# =========================
# 16. Plot 5: region-level barplot
# =========================
x = np.arange(len(region_summary_df))

fig, ax = plt.subplots(figsize=(18, 6))
ax.bar(x - 0.2, region_summary_df["Residue_entropy_mean"], width=0.4, label="Residue entropy mean")
ax.bar(x + 0.2, region_summary_df["Property_entropy_mean"], width=0.4, label="Property entropy mean")
ax.set_xticks(x)
ax.set_xticklabels(region_summary_df["Functional_region"], rotation=60, ha="right")
ax.set_ylabel("Mean entropy")
ax.set_title("Mean entropy by KLIFS functional region")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

plot5_path = os.path.join(output_dir, "functional_region_entropy_barplot.png")
plt.savefig(plot5_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved plot: {plot5_path}")

# =========================
# 17. Console summary
# =========================
print("\n===== Functional region summary =====")
print(region_summary_df.to_string(index=False))

print("\nTop 10 lowest residue entropy positions:")
print(
    residue_entropy_df[["KLIFS_position", "Functional_region", "Residue_entropy", "Most_common_residue", "Most_common_residue_freq"]]
    .sort_values("Residue_entropy", ascending=True)
    .head(10)
    .to_string(index=False)
)

print("\nTop 10 highest residue entropy positions:")
print(
    residue_entropy_df[["KLIFS_position", "Functional_region", "Residue_entropy", "Unique_residue_count"]]
    .sort_values("Residue_entropy", ascending=False)
    .head(10)
    .to_string(index=False)
)

print("\nTop 10 lowest property mean entropy positions:")
print(
    property_entropy_position_df[["KLIFS_position", "Functional_region", "Property_entropy_mean", "Property_entropy_std"]]
    .sort_values("Property_entropy_mean", ascending=True)
    .head(10)
    .to_string(index=False)
)

print("\nTop 10 highest property mean entropy positions:")
print(
    property_entropy_position_df[["KLIFS_position", "Functional_region", "Property_entropy_mean", "Property_entropy_std"]]
    .sort_values("Property_entropy_mean", ascending=False)
    .head(10)
    .to_string(index=False)
)

print("\nAll output saved to:")
print(output_dir)

Loaded rows: 482
Columns: ['kinase_name', 'UniProt_ID', 'KLIFS', 'Binary_string']
Rows retained for KLIFS analysis: 482
Saved residue entropy table: ./entropy_analysis_output/residue_entropy_by_KLIFS_position.csv
Rows retained for property analysis: 482
Saved property entropy summary: ./entropy_analysis_output/property_entropy_summary_by_KLIFS_position.csv
Saved property entropy detail: ./entropy_analysis_output/property_entropy_detailed_by_KLIFS_position.csv
Saved merged summary: ./entropy_analysis_output/residue_property_entropy_merged.csv
Saved functional region summary: ./entropy_analysis_output/functional_region_entropy_summary.csv
Saved plot: ./entropy_analysis_output/residue_entropy_plot_with_regions.png
Saved plot: ./entropy_analysis_output/residue_entropy_sorted_descending.png
Saved plot: ./entropy_analysis_output/property_mean_entropy_plot_with_regions.png
Saved plot: ./entropy_analysis_output/residue_vs_property_entropy_plot_with_regions.png
Saved plot: ./entropy_analysis_ou